# Lab 2 — Gradient Descent và Linear Regression từ đầu

**Thời lượng:** 120–150 phút  
**Mục tiêu:** cài MSE/gradient, kiểm tra gradient, thử learning rate, chuẩn hóa feature và đối chiếu với `np.linalg.lstsq`.

> Quy tắc lab: chạy **Restart Kernel → Run All**. Không tiếp tục huấn luyện nếu gradient check chưa đạt `relative_error < 1e-6`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
FIGURES = ROOT / 'outputs' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
np.set_printoptions(precision=6, suppress=True)
print('Project root:', ROOT.resolve())

## 1. Xây các khối cơ bản

Với $m$ mẫu, design matrix có bias là $X_b\in\mathbb{R}^{m\times(n+1)}$. Ta dùng:

$$\hat y=X_b\theta,\qquad J(\theta)=\frac{1}{m}\lVert X_b\theta-y\rVert_2^2,$$

$$\nabla_\theta J=\frac{2}{m}X_b^T(X_b\theta-y).$$

In [ ]:
def add_bias(X):
    X = np.asarray(X, dtype=float)
    return np.column_stack([np.ones(X.shape[0]), X])


def predict(X_bias, theta):
    return X_bias @ theta


def mse(y, y_hat):
    return float(np.mean((y_hat - y) ** 2))


def mse_gradient(X_bias, y, theta):
    residual = predict(X_bias, theta) - y
    return (2.0 / len(y)) * (X_bias.T @ residual)


def numerical_gradient(loss_fn, theta, epsilon=1e-6):
    result = np.empty_like(theta, dtype=float)
    for j in range(theta.size):
        plus, minus = theta.copy(), theta.copy()
        plus[j] += epsilon
        minus[j] -= epsilon
        result[j] = (loss_fn(plus) - loss_fn(minus)) / (2 * epsilon)
    return result


def relative_error(a, b):
    return np.linalg.norm(a - b) / max(1.0, np.linalg.norm(a) + np.linalg.norm(b))

## 2. Gradient check bằng central finite differences

Numerical gradient chỉ là **oracle để kiểm tra**, không được dùng làm gradient huấn luyện. Thử nhiều `epsilon`: quá lớn gây approximation error, quá nhỏ gây floating-point cancellation.

In [ ]:
housing = pd.read_csv(ROOT / 'data' / 'housing_regression.csv')
feature_names = ['area_sqm', 'bedrooms', 'age_years', 'distance_km', 'energy_score']
X_small = housing.loc[:11, feature_names].to_numpy(dtype=float)
y_small = housing.loc[:11, 'price_thousand'].to_numpy(dtype=float)
mean_small = X_small.mean(axis=0)
scale_small = X_small.std(axis=0)
Xb_small = add_bias((X_small - mean_small) / scale_small)
theta_probe = np.linspace(-0.3, 0.3, Xb_small.shape[1])
analytical = mse_gradient(Xb_small, y_small, theta_probe)

for epsilon in [1e-4, 1e-5, 1e-6, 1e-7]:
    numerical = numerical_gradient(
        lambda t: mse(y_small, predict(Xb_small, t)), theta_probe, epsilon
    )
    print(f'epsilon={epsilon:.0e}  relative_error={relative_error(analytical, numerical):.3e}')

numerical = numerical_gradient(
    lambda t: mse(y_small, predict(Xb_small, t)), theta_probe, 1e-6
)
gradient_check_error = relative_error(analytical, numerical)
assert gradient_check_error < 1e-6

## 3. Full-batch gradient descent

Lưu cả loss ban đầu và loss sau từng bước. Dừng ngay khi loss không hữu hạn; đây thường là dấu hiệu learning rate quá lớn hoặc feature chưa scale.

In [ ]:
def gradient_descent(X_bias, y, learning_rate, n_steps, initial_theta=None):
    theta = (
        np.zeros(X_bias.shape[1], dtype=float)
        if initial_theta is None
        else np.asarray(initial_theta, dtype=float).copy()
    )
    losses = np.empty(n_steps + 1, dtype=float)
    for step in range(n_steps + 1):
        with np.errstate(over='ignore', invalid='ignore'):
            current_loss = mse(y, predict(X_bias, theta))
        if not np.isfinite(current_loss):
            raise FloatingPointError('Loss diverged; reduce learning rate or scale features.')
        losses[step] = current_loss
        if step < n_steps:
            theta -= learning_rate * mse_gradient(X_bias, y, theta)
    return theta, losses

## 4. Warm-up: hồi quy một feature

So sánh gradient descent với nghiệm least-squares ổn định từ `np.linalg.lstsq`.

In [ ]:
single = pd.read_csv(ROOT / 'data' / 'single_feature_regression.csv')
x = single[['x']].to_numpy(dtype=float)
y_single = single['y'].to_numpy(dtype=float)
Xb_single = add_bias(x)
theta_single, loss_single = gradient_descent(
    Xb_single, y_single, learning_rate=0.01, n_steps=4_000
)
theta_lstsq_single = np.linalg.lstsq(Xb_single, y_single, rcond=None)[0]
print('GD theta:    ', theta_single)
print('lstsq theta: ', theta_lstsq_single)
print(f'MSE: {loss_single[-1]:.6f}')
assert np.allclose(theta_single, theta_lstsq_single, atol=1e-5)

In [ ]:
order = np.argsort(x[:, 0])
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x[:, 0], y_single, alpha=0.65, label='Dữ liệu')
ax.plot(x[order, 0], predict(Xb_single, theta_single)[order], color='crimson', label='GD fit')
ax.set(xlabel='x', ylabel='y', title='Linear Regression — một feature')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / 'fitted_line.png', dpi=150)
plt.show()

## 5. Controlled experiment: learning rate

Chỉ thay đổi learning rate; giữ nguyên dữ liệu, khởi tạo và số bước. Feature được standardize để các hướng trong loss surface có scale tương đồng hơn.

In [ ]:
X = housing[feature_names].to_numpy(dtype=float)
y = housing['price_thousand'].to_numpy(dtype=float)
feature_mean = X.mean(axis=0)
feature_scale = X.std(axis=0)
X_scaled = (X - feature_mean) / feature_scale
Xb = add_bias(X_scaled)

learning_rates = [0.001, 0.01, 0.05, 0.2, 1.0]
experiment_rows = []
histories = {}
for lr in learning_rates:
    try:
        theta_lr, history_lr = gradient_descent(Xb, y, lr, 500)
        histories[lr] = history_lr
        stable = bool(np.all(np.diff(history_lr) <= 1e-8) and history_lr[-1] < history_lr[0])
        experiment_rows.append({
            'learning_rate': lr, 'initial_mse': history_lr[0],
            'final_mse': history_lr[-1], 'stable': stable
        })
    except FloatingPointError:
        experiment_rows.append({
            'learning_rate': lr, 'initial_mse': np.nan,
            'final_mse': np.inf, 'stable': False
        })

experiment = pd.DataFrame(experiment_rows)
experiment.to_csv(ROOT / 'outputs' / 'learning_rate_experiment.csv', index=False)
experiment

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for lr, history in histories.items():
    ax.plot(history, label=f'lr={lr:g}')
ax.set_yscale('log')
ax.set(xlabel='Bước cập nhật', ylabel='MSE (log scale)', title='Ảnh hưởng của learning rate')
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES / 'loss_by_learning_rate.png', dpi=150)
plt.show()

## 6. Mô hình cuối và đổi hệ số về đơn vị gốc

Nếu $z_j=(x_j-\mu_j)/s_j$ và mô hình học trên $z$, thì $w_j=\theta_j/s_j$ và $b=\theta_0-w^T\mu$.

In [ ]:
theta_scaled, final_history = gradient_descent(Xb, y, learning_rate=0.05, n_steps=2_000)
coef_raw = theta_scaled[1:] / feature_scale
intercept_raw = theta_scaled[0] - coef_raw @ feature_mean
predictions = intercept_raw + X @ coef_raw
theta_lstsq = np.linalg.lstsq(add_bias(X), y, rcond=None)[0]

comparison = pd.DataFrame({
    'parameter': ['intercept', *feature_names],
    'gradient_descent': np.r_[intercept_raw, coef_raw],
    'lstsq': theta_lstsq,
})
comparison['absolute_difference'] = abs(comparison['gradient_descent'] - comparison['lstsq'])
print(comparison.to_string(index=False))
print(f'Final housing MSE: {mse(y, predictions):.6f}')
assert np.allclose(np.r_[intercept_raw, coef_raw], theta_lstsq, atol=1e-5)
assert abs(mse(y, predictions) - 137.069193146) < 1e-6

In [ ]:
prediction_frame = pd.DataFrame({
    'property_id': housing['property_id'],
    'actual_price_thousand': y,
    'predicted_price_thousand': predictions,
    'residual': predictions - y,
})
prediction_frame.to_csv(ROOT / 'outputs' / 'housing_predictions.csv', index=False)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y, predictions, alpha=0.65)
limits = [min(y.min(), predictions.min()), max(y.max(), predictions.max())]
ax.plot(limits, limits, '--', color='black', label='Dự đoán hoàn hảo')
ax.set(xlabel='Actual price_thousand', ylabel='Predicted price_thousand', title='Predicted vs Actual')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / 'predicted_vs_actual.png', dpi=150)
plt.show()

## 7. Ablation: bỏ standardization

Dữ liệu và số bước giữ nguyên. Learning rate rất nhỏ có thể chạy nhưng chậm; learning rate dùng tốt trên dữ liệu đã scale thường làm nghiệm không scale mất ổn định.

In [ ]:
Xb_unscaled = add_bias(X)
unscaled_results = []
for lr in [1e-7, 1e-6, 1e-5, 1e-4]:
    try:
        _, history = gradient_descent(Xb_unscaled, y, lr, 1_000)
        unscaled_results.append((lr, history[-1], 'finite'))
    except FloatingPointError:
        unscaled_results.append((lr, np.inf, 'diverged'))
pd.DataFrame(unscaled_results, columns=['learning_rate', 'final_mse', 'status'])

## Exit ticket

1. Vì sao numerical gradient không nên dùng để train?
2. Dựa trên loss curves, chọn learning rate nào và nêu bằng chứng.
3. Standardization có đổi nghiệm tối ưu trong đơn vị gốc không? Nó đổi điều gì?
4. Phân biệt lỗi gradient sai với learning rate quá lớn bằng hai thí nghiệm nào?

In [ ]:
assert gradient_check_error < 1e-6
assert final_history[-1] < final_history[0] * 0.01
assert np.isfinite(predictions).all()
assert (FIGURES / 'loss_by_learning_rate.png').exists()
print('PASS — gradient, convergence, lstsq parity và outputs đã được kiểm tra.')